# PathSnap preprocessing and feature engineering

This local Python/Jupyter notebook builds model-ready candidate features from the audited raw trips. It leaves raw files unchanged and writes derived files to datasets/processed_v1.

The current local data contains 80 training trips and 10 student_test trips. The evaluator partition is not present in the project tree, so it is not scanned.

## Label and leakage handling
- Training route files are used only to align each GPS observation to a likely route arc and create the training target. Route IDs and reference routes are never candidate prediction features.
- Student-test route files are not opened. Student-test outputs contain features only.
- GPS observations produce nearby road-arc candidates. Training candidates are labelled positive when their arc matches the aligned route arc; candidate-miss observations are recorded in the manifest and excluded from the candidate training table rather than incorrectly labelled negative.
- Route-to-observation alignment is a monotonic, distance-based dynamic program. It is an initial label transfer assumption; review its candidate coverage and route anomalies before model fitting.
- Features include point-to-segment distance, segment projection, heading difference, local speed/time gap, arc length and endpoint degrees. Coordinates are projected into a local azimuthal-equidistant metric frame per trip.

**Important:** Candidate miss rate measures whether the aligned reference arc appears among the generated candidates. High miss rates limit achievable accuracy regardless of classifier quality.

In [1]:
from pathlib import Path
import json, gzip, math, time, warnings
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree
from IPython.display import display

ROOT = Path(r'D:\Projects\PathSnap_ML_Hackathon')
PARTITIONS = {'training': ROOT / 'train', 'student_testing': ROOT / 'student_test'}
OUTPUT_DIR = ROOT / 'datasets' / 'processed_v1'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEARCH_RADIUS_M = 100.0
MAX_CANDIDATES = 20
CANDIDATE_QUERY_K = 128
ROUTE_STATES = 16
ROUTE_QUERY_K = 96
ROAD_SAMPLE_SPACING_M = 40.0
EARTH_RADIUS_M = 6_371_008.8
print('Project root:', ROOT, '| exists:', ROOT.exists())
for name, folder in PARTITIONS.items():
    print(name, folder, 'exists:', folder.exists(), 'trip folders:', len(list(folder.iterdir())) if folder.exists() else 0)

Project root: D:\Projects\PathSnap_ML_Hackathon | exists: True
training D:\Projects\PathSnap_ML_Hackathon\train exists: True trip folders: 80
student_testing D:\Projects\PathSnap_ML_Hackathon\student_test exists: True trip folders: 10


In [2]:
def read_table(path, columns):
    arr = np.loadtxt(path, dtype=float, ndmin=2)
    if arr.size == 0:
        return pd.DataFrame(columns=columns, dtype=float)
    if arr.shape[1] != len(columns):
        raise ValueError(f'{path}: expected {len(columns)} columns, found {arr.shape[1]}')
    return pd.DataFrame(arr, columns=columns)

def project_lonlat(lon, lat, lon0, lat0):
    # Spherical azimuthal-equidistant projection centred on this trip.
    lon=np.radians(np.asarray(lon,dtype=float)); lat=np.radians(np.asarray(lat,dtype=float))
    lon0r,lat0r=np.radians(lon0),np.radians(lat0); dlon=lon-lon0r
    cos_c=np.sin(lat0r)*np.sin(lat)+np.cos(lat0r)*np.cos(lat)*np.cos(dlon)
    c=np.arccos(np.clip(cos_c,-1.0,1.0)); sin_c=np.sin(c)
    k=np.ones_like(c); mask=np.abs(sin_c)>1e-12; k[mask]=c[mask]/sin_c[mask]
    x=EARTH_RADIUS_M*k*np.cos(lat)*np.sin(dlon)
    y=EARTH_RADIUS_M*k*(np.cos(lat0r)*np.sin(lat)-np.sin(lat0r)*np.cos(lat)*np.cos(dlon))
    return np.column_stack([x,y])

def segment_geometry(nodes_xy, arcs):
    n_nodes=len(nodes_xy); arc_ids=np.arange(len(arcs),dtype=np.int64)
    endpoints=arcs.astype(np.int64)
    valid=(np.isfinite(arcs).all(axis=1)&(arcs==endpoints).all(axis=1)&(endpoints[:,0]>=0)&(endpoints[:,1]>=0)&(endpoints[:,0]<n_nodes)&(endpoints[:,1]<n_nodes))
    valid_ids=arc_ids[valid]; uv=endpoints[valid]
    p1=nodes_xy[uv[:,0]]; p2=nodes_xy[uv[:,1]]; vec=p2-p1; lengths=np.linalg.norm(vec,axis=1)
    bearings=(np.degrees(np.arctan2(vec[:,0],vec[:,1]))+360)%360
    return {'arc_id':valid_ids,'u':uv[:,0],'v':uv[:,1],'p1':p1,'p2':p2,'length':lengths,'bearing':bearings,'invalid_count':int((~valid).sum()),'n_nodes':n_nodes}

def sampled_tree(segments, spacing=ROAD_SAMPLE_SPACING_M):
    points=[]; owners=[]
    for i,(a,b,length) in enumerate(zip(segments['p1'],segments['p2'],segments['length'])):
        n=max(1,int(np.ceil(length/spacing)))
        ts=np.linspace(0.0,1.0,n+1)
        points.append(a[None,:]+ts[:,None]*(b-a)[None,:])
        owners.extend([i]*len(ts))
    pts=np.vstack(points) if points else np.empty((0,2))
    return (cKDTree(pts) if len(pts) else None),np.asarray(owners,dtype=np.int64),pts

def exact_candidates(query_xy, tree, owners, sample_points, segments, k, max_distance=None):
    if tree is None or not len(segments['arc_id']):
        return [[] for _ in range(len(query_xy))]
    kk=min(k,len(sample_points)); _,sample_idx=tree.query(query_xy,k=kk,workers=-1)
    if kk==1: sample_idx=sample_idx[:,None]
    out=[]
    for point,si in zip(query_xy,sample_idx):
        seg_idx=np.unique(owners[np.asarray(si,dtype=np.int64)])
        a=segments['p1'][seg_idx]; b=segments['p2'][seg_idx]; v=b-a; den=np.einsum('ij,ij->i',v,v)
        f=np.divide(np.einsum('ij,ij->i',(point-a),v),den,out=np.zeros_like(den),where=den>0); f=np.clip(f,0,1)
        proj=a+f[:,None]*v; dist=np.linalg.norm(point-proj,axis=1)
        order=np.argsort(dist,kind='stable')
        if max_distance is not None:
            within=order[dist[order]<=max_distance]
            if len(within): order=within
            else: order=order[:min(3,len(order))]  # retain a fallback candidate, but measure coverage
        order=order[:MAX_CANDIDATES if max_distance==SEARCH_RADIUS_M else k]
        out.append([(int(seg_idx[j]),float(dist[j]),float(f[j])) for j in order])
    return out

def angle_difference(a,b):
    d=np.abs(a-b)%360
    return np.minimum(d,360-d)

def align_route_to_track(track_xy, track_step, route_segments):
    n=len(track_xy); target=np.full(n,-1,dtype=np.int64); choices=[[] for _ in range(n)]
    if not n or not len(route_segments['arc_id']): return target,0
    tree,owners,samples=sampled_tree(route_segments)
    choices=exact_candidates(track_xy,tree,owners,samples,route_segments,ROUTE_QUERY_K,max_distance=250.0)
    starts=np.r_[0.0,np.cumsum(route_segments['length'][:-1])]
    state_rows=[]
    for row in choices:
        states=[]
        for seg_i,dist,frac in row[:ROUTE_STATES]:
            states.append((seg_i,starts[seg_i]+frac*route_segments['length'][seg_i],dist))
        state_rows.append(states)
    back=np.full((n,ROUTE_STATES),-1,dtype=np.int16); prev_cost=None; prev_prog=None
    for i,states in enumerate(state_rows):
        if not states: prev_cost=None;prev_prog=None;continue
        prog=np.array([s[1] for s in states]); emission=np.array([s[2]/20.0 for s in states])
        if i==0 or prev_cost is None:
            cost=emission.copy()
        else:
            cost=np.full(len(states),np.inf)
            for j,p in enumerate(prog):
                allowed=np.flatnonzero(prev_prog<=p+1e-6)
                if not len(allowed):
                    allowed=np.arange(len(prev_prog)); trans=np.full(len(allowed),20.0)
                else:
                    trans=np.minimum(np.abs((p-prev_prog[allowed])-max(float(track_step[i]),0.0))/60.0,25.0)
                vals=prev_cost[allowed]+trans
                k=int(np.argmin(vals));cost[j]=emission[j]+vals[k];back[i,j]=int(allowed[k])
        prev_cost=cost;prev_prog=prog
    if prev_cost is None:return target,0
    state_idx=int(np.argmin(prev_cost))
    for i in range(n-1,-1,-1):
        states=state_rows[i]
        if not states: continue
        state_idx=min(state_idx,len(states)-1); seg_i=states[state_idx][0]
        target[i]=int(route_segments['arc_id'][seg_i])
        state_idx=int(back[i,state_idx]) if i>0 and back[i,state_idx]>=0 else 0
    return target,int(np.sum(target>=0))

## Build features and labels

The processing loop uses only the training route files for label construction. Student-testing data are processed without reading their route files. Candidate generation uses spatially sampled road segments so long arcs remain searchable, then computes exact point-to-segment distances. The output is candidate-level, which is the appropriate shape for training a candidate-scoring classifier.

In [3]:
CANDIDATE_COLUMNS=['trip_id','observation_idx','source_line','timestamp_s','gps_longitude','gps_latitude','gps_step_m','gps_speed_mps','time_delta_s','gps_heading_deg','gps_heading_change_deg','candidate_rank','candidate_arc_id','candidate_distance_m','projection_fraction','arc_length_m','arc_bearing_deg','heading_difference_deg','from_node_out_degree','from_node_in_degree','to_node_out_degree','to_node_in_degree','is_target']
OBS_COLUMNS=['trip_id','observation_idx','source_line','timestamp_s','gps_longitude','gps_latitude','gps_step_m','gps_speed_mps','time_delta_s','gps_heading_deg','gps_heading_change_deg','aligned_target_arc_id','candidate_hit']
manifest=[]; started=time.time(); first_candidate={'training':True,'student_testing':True};first_obs={'training':True,'student_testing':True}

with gzip.open(OUTPUT_DIR/'training_candidates.csv.gz','wt',newline='',encoding='utf-8') as train_out, \
     gzip.open(OUTPUT_DIR/'student_test_candidates.csv.gz','wt',newline='',encoding='utf-8') as test_out, \
     gzip.open(OUTPUT_DIR/'training_observations.csv.gz','wt',newline='',encoding='utf-8') as train_obs_out, \
     gzip.open(OUTPUT_DIR/'student_test_observations.csv.gz','wt',newline='',encoding='utf-8') as test_obs_out:
    for partition,folder in PARTITIONS.items():
        if not folder.exists(): raise FileNotFoundError(folder)
        trips=sorted(p for p in folder.iterdir() if p.is_dir())
        for trip_dir in trips:
            trip=trip_dir.name; prefix=trip_dir/trip
            gps=read_table(str(prefix)+'.track',['longitude','latitude','timestamp_s'])
            nodes=read_table(str(prefix)+'.nodes',['longitude','latitude'])
            arcs=read_table(str(prefix)+'.arcs',['from_node_id','to_node_id']).to_numpy()
            gps=gps.replace([np.inf,-np.inf],np.nan).dropna().reset_index(drop=True)
            gps=gps[gps.longitude.between(-180,180)&gps.latitude.between(-90,90)].reset_index(drop=True)
            if gps.empty or nodes.empty:
                manifest.append({'trip_id':trip,'partition':partition,'status':'empty_gps_or_network','gps_rows':len(gps)});continue
            lon0=float(gps.longitude.mean());lat0=float(gps.latitude.mean())
            gps_xy=project_lonlat(gps.longitude.to_numpy(),gps.latitude.to_numpy(),lon0,lat0)
            nodes_xy=project_lonlat(nodes.longitude.to_numpy(),nodes.latitude.to_numpy(),lon0,lat0)
            seg=segment_geometry(nodes_xy,arcs);net_tree,net_owner,net_samples=sampled_tree(seg)
            step=np.zeros(len(gps));dt=np.full(len(gps),np.nan);speed=np.full(len(gps),np.nan);heading=np.full(len(gps),np.nan);turn=np.full(len(gps),np.nan)
            if len(gps)>1:
                delta=np.diff(gps_xy,axis=0);step[1:]=np.linalg.norm(delta,axis=1);dt[1:]=np.diff(gps.timestamp_s.to_numpy())
                good=(dt[1:]>0);speed[1:][good]=step[1:][good]/dt[1:][good]
                valid_heading=(step[1:]>=1)&good;hh=(np.degrees(np.arctan2(delta[:,0],delta[:,1]))+360)%360;heading[1:][valid_heading]=hh[valid_heading]
                for i in range(2,len(gps)):
                    if np.isfinite(heading[i]) and np.isfinite(heading[i-1]):turn[i]=angle_difference(heading[i],heading[i-1])
            cand=exact_candidates(gps_xy,net_tree,net_owner,net_samples,seg,CANDIDATE_QUERY_K,max_distance=SEARCH_RADIUS_M)
            target=np.full(len(gps),-1,dtype=np.int64); route_invalid=0;route_breaks=0;aligned=0
            if partition=='training':
                route_path=str(prefix)+'.route'
                if not Path(route_path).exists():raise FileNotFoundError(f'Missing training route: {route_path}')
                route_ids=read_table(route_path,['arc_id']).arc_id.to_numpy()
                valid_route=np.isfinite(route_ids)&(route_ids==np.floor(route_ids))&(route_ids>=0)&(route_ids<len(arcs))
                route_invalid=int((~valid_route).sum());route_ids=route_ids[valid_route].astype(np.int64)
                id_to_seg={int(aid):i for i,aid in enumerate(seg['arc_id'])}
                route_seg_idx=[id_to_seg[int(aid)] for aid in route_ids if int(aid) in id_to_seg]
                if route_seg_idx:
                    route_seg={k:seg[k][route_seg_idx] for k in ['arc_id','u','v','p1','p2','length','bearing']}
                    route_breaks=int(np.sum(route_seg['v'][:-1]!=route_seg['u'][1:])) if len(route_seg['arc_id'])>1 else 0
                    target,aligned=align_route_to_track(gps_xy,step,route_seg)
            target_hit=np.zeros(len(gps),dtype=bool)
            if partition=='training':
                for i,rows in enumerate(cand):target_hit[i]=target[i]>=0 and any(int(seg['arc_id'][j])==target[i] for j,_,_ in rows)
            obs_df=pd.DataFrame({'trip_id':trip,'observation_idx':np.arange(len(gps)),'source_line':np.arange(1,len(gps)+1),'timestamp_s':gps.timestamp_s,'gps_longitude':gps.longitude,'gps_latitude':gps.latitude,'gps_step_m':step,'gps_speed_mps':speed,'time_delta_s':dt,'gps_heading_deg':heading,'gps_heading_change_deg':turn})
            obs_df['aligned_target_arc_id']=target if partition=='training' else np.nan
            obs_df['candidate_hit']=target_hit if partition=='training' else np.nan
            obs_df.to_csv(train_obs_out if partition=='training' else test_obs_out,index=False,header=first_obs[partition]);first_obs[partition]=False
            rows=[];candidate_hits=int(target_hit.sum()) if partition=='training' else 0;candidate_misses=int(np.sum((target>=0)&(~target_hit))) if partition=='training' else 0;unaligned=int(np.sum(target<0)) if partition=='training' else 0
            outdeg=np.bincount(seg['u'],minlength=len(nodes)); indeg=np.bincount(seg['v'],minlength=len(nodes))
            for i,options in enumerate(cand):
                if partition=='training' and (target[i]<0 or not target_hit[i]):continue
                for rank,(si,distance,fraction) in enumerate(options,1):
                    aid=int(seg['arc_id'][si]);is_target=int(partition=='training' and aid==target[i])
                    rows.append([trip,i,i+1,float(gps.timestamp_s.iloc[i]),float(gps.longitude.iloc[i]),float(gps.latitude.iloc[i]),float(step[i]),speed[i],dt[i],heading[i],turn[i],rank,aid,distance,fraction,float(seg['length'][si]),float(seg['bearing'][si]),float(angle_difference(heading[i],seg['bearing'][si])) if np.isfinite(heading[i]) else np.nan,int(outdeg[seg['u'][si]]),int(indeg[seg['u'][si]]),int(outdeg[seg['v'][si]]),int(indeg[seg['v'][si]]),is_target if partition=='training' else np.nan])
            cand_df=pd.DataFrame(rows,columns=CANDIDATE_COLUMNS)
            if partition=='student_testing':cand_df=cand_df.drop(columns='is_target')
            cand_df.to_csv(train_out if partition=='training' else test_out,index=False,header=first_candidate[partition]);first_candidate[partition]=False
            manifest.append({'trip_id':trip,'partition':partition,'status':'processed','gps_rows':len(gps),'nodes':len(nodes),'arcs':len(arcs),'invalid_arc_rows':seg['invalid_count'],'route_arcs':len(route_ids) if partition=='training' else None,'invalid_route_rows':route_invalid,'route_disconnected_joins':route_breaks,'aligned_observations':aligned if partition=='training' else None,'unaligned_observations':unaligned if partition=='training' else None,'candidate_feature_rows':len(cand_df),'candidate_hit_observations':candidate_hits if partition=='training' else None,'candidate_miss_observations':candidate_misses if partition=='training' else None,'candidate_miss_rate':candidate_misses/(candidate_hits+candidate_misses) if candidate_hits+candidate_misses else None,'end_to_end_candidate_coverage':candidate_hits/len(gps) if partition=='training' and len(gps) else None})
            print(f'{partition} {trip}: {len(gps):,} observations, {len(cand_df):,} candidate rows'+(f', candidate miss {candidate_misses/(candidate_hits+candidate_misses):.1%}' if candidate_hits+candidate_misses else ''))

manifest_df=pd.DataFrame(manifest).sort_values(['partition','trip_id'])
manifest_df.to_csv(OUTPUT_DIR/'trip_manifest.csv',index=False)
manifest_df.to_json(OUTPUT_DIR/'trip_manifest.json',orient='records',indent=2)
print(f'\nProcessed {len(manifest_df)} trips in {(time.time()-started)/60:.1f} minutes. Outputs: {OUTPUT_DIR}')
display(manifest_df.groupby('partition').agg(trips=('trip_id','nunique'),gps_rows=('gps_rows','sum'),candidate_rows=('candidate_feature_rows','sum'),mean_candidate_miss_rate=('candidate_miss_rate','mean')))

training 00000001: 1,315 observations, 21,858 candidate rows, candidate miss 0.0%
training 00000002: 7,243 observations, 99,670 candidate rows, candidate miss 10.6%
training 00000003: 821 observations, 15,620 candidate rows, candidate miss 1.5%
training 00000004: 1,064 observations, 12,168 candidate rows, candidate miss 1.7%
training 00000005: 1,045 observations, 4,783 candidate rows, candidate miss 0.0%
training 00000006: 1,450 observations, 21,408 candidate rows, candidate miss 0.1%
training 00000008: 3,093 observations, 58,969 candidate rows, candidate miss 3.7%
training 00000010: 2,711 observations, 50,009 candidate rows, candidate miss 3.6%
training 00000011: 2,828 observations, 26,929 candidate rows, candidate miss 0.1%
training 00000012: 1,608 observations, 10,208 candidate rows, candidate miss 0.0%
training 00000013: 2,580 observations, 36,130 candidate rows, candidate miss 0.0%
training 00000014: 448 observations, 8,552 candidate rows, candidate miss 0.0%
training 00000016: 1,

,trips,gps_rows,candidate_rows,mean_candidate_miss_rate
partition,,,,
student_testing,10,19160,275598,NaN
training,80,208417,2400632,0.040187


## Review before model training

- Confirm the manifest has 80 training trips and 10 student-testing trips. Evaluator testing is intentionally absent.
- Sort training trips by candidate_miss_rate. High-miss trips need candidate-radius/search review or label-alignment inspection before model fitting.
- Check route_disconnected_joins; these are present in the supplied audit and should not be silently repaired.
- Use trip_id for group-aware validation. Fit any scaling/encoding inside each training fold only. Do not use aligned_target_arc_id, candidate_hit, or is_target as input features.
- Student-test route files were not read, and student-test candidates contain no is_target field. Keep reference labels separate from the prediction pipeline.

In [4]:
print('Output files:')
for p in sorted(OUTPUT_DIR.iterdir()):print(p.name, f'{p.stat().st_size/1e6:.2f} MB')
print('Training candidate columns:');print(pd.read_csv(OUTPUT_DIR/'training_candidates.csv.gz',nrows=3).columns.tolist())
print('Student-test candidate columns:');print(pd.read_csv(OUTPUT_DIR/'student_test_candidates.csv.gz',nrows=3).columns.tolist())

Output files:
student_test_candidates.csv.gz 8.92 MB
student_test_observations.csv.gz 0.85 MB
training_candidates.csv.gz 78.20 MB
training_observations.csv.gz 9.67 MB
trip_manifest.csv 0.01 MB
trip_manifest.json 0.05 MB
Training candidate columns:
['trip_id', 'observation_idx', 'source_line', 'timestamp_s', 'gps_longitude', 'gps_latitude', 'gps_step_m', 'gps_speed_mps', 'time_delta_s', 'gps_heading_deg', 'gps_heading_change_deg', 'candidate_rank', 'candidate_arc_id', 'candidate_distance_m', 'projection_fraction', 'arc_length_m', 'arc_bearing_deg', 'heading_difference_deg', 'from_node_out_degree', 'from_node_in_degree', 'to_node_out_degree', 'to_node_in_degree', 'is_target']
Student-test candidate columns:
['trip_id', 'observation_idx', 'source_line', 'timestamp_s', 'gps_longitude', 'gps_latitude', 'gps_step_m', 'gps_speed_mps', 'time_delta_s', 'gps_heading_deg', 'gps_heading_change_deg', 'candidate_rank', 'candidate_arc_id', 'candidate_distance_m', 'projection_fraction', 'arc_length_m'